# Antimicrobial Susceptibility Data Analysis

This notebook processes AST data to calculate mean inhibition zone diameters, standard deviations, and generate visualization plots.

## Workflow
1. Load AST dataset
2. Clean and reshape data
3. Calculate mean and SD values
4. Generate visualization plots


# Load AST data with two-level headers:
Level 1 = Antibiotic name

Level 2 = Measurement type (Mean, SD)

In [ ]:
import pandas as pd

df_raw = pd.read_excel(
    "data/AST_Data.xlsx",
    header=[0, 1]
)

df_raw.head()

## Flattening Multi-Level Column Headers

The AST dataset contains hierarchical (multi-level) column headers because the Excel file uses two header rows:

- Level 1: Antibiotic names
- Level 2: Measurement type: Mean, SD

To facilitate data processing and visualization, the multi-level column structure is converted into a single-level format by combining both header levels into unified column names (e.g., `AMP_Mean`, `AMP_SD`).

This step ensures compatibility with downstream reshaping and plotting operations.

In [ ]:
df_raw.columns = [
    f"{a}_{b}" if b else a
    for a, b in df_raw.columns
]

df_raw.head()

## Reshaping the Dataset into Long Format

The original dataset is structured in wide format, where each antibiotic and its associated metrics (Mean, SD) occupy separate columns.

To enable statistical analysis and visualization, the dataset is transformed into a long (tidy) format using a melting operation:

- Identifier variables (`id_vars`) such as bacterial species and strain numbers are preserved.
- Antibiotic and metric information are combined into a single column.
- Corresponding measurement values are stored in a unified value column.

Additionally, the first row is removed because it contains residual header artifacts from the Excel formatting.

This transformation produces an analysis-ready dataset suitable for plotting and comparative analyses.


In [ ]:
# Define ID columns
id_vars = [
    "Bacteria Name_Unnamed: 0_level_1",
    "Strain Number_Unnamed: 1_level_1"
]

# Drop the first row which contains NaN for the id_vars and is part of the header.
df_raw_cleaned = df_raw.drop(0).copy()


# Melt the dataframe
df_long = (
    df_raw_cleaned
    .melt(
        id_vars=id_vars,
        var_name="Antibiotic_Metric",
        value_name="Value"
    )
)


## Parsing Antibiotic and Metric Information

After reshaping the dataset into long format, the column `Antibiotic_Metric` contains combined information representing both:

- The antibiotic name
- The associated measurement type (e.g., Mean, SD, AST)

To improve clarity and enable structured analysis, this combined field is parsed into two separate columns:

- `Antibiotic` – indicating the name
- `Metric` – indicating the type of measurement

A helper function is defined to:

1. Remove any unwanted column prefixes
2. Identify the metric type based on suffix patterns (e.g., `.1`, `.2`)
3. Extract the antibiotic name accordingly

The resulting structured columns replace the original combined field, producing a fully tidy dataset suitable for downstream statistical analysis and visualization.


In [ ]:
# Re-create df_long to ensure 'Antibiotic_Metric' is present and correctly processed
# Take a combined column name and returns: Antibiotic name and Metric type

def get_antibiotic_and_metric(col_name):


        # Remove 'Strain Number_' prefix if present
    if col_name.startswith("Strain Number_"):
        col_name = col_name.replace("Strain Number_", "")

        # Determine metric based on suffix
    if col_name.endswith(".1"):
        antibiotic = col_name.replace(".1", "").strip()
        metric = "SD"
    elif col_name.endswith(".2"):
        antibiotic = col_name.replace(".2", "").strip()
        metric = "AST"    # AST interpretations are Resistant (R), Susceptible (S), Intermediate (I)
    else:
        # Assuming no suffix implies 'Mean'
        antibiotic = col_name.strip()
        metric = "Mean"

        # Return structured output
    return antibiotic, metric

# Apply the function to create new temporary columns
df_long[["Antibiotic", "Metric"]] = df_long["Antibiotic_Metric"].apply(lambda x: pd.Series(get_antibiotic_and_metric(x)))

# Drop the original 'Antibiotic_Metric' column
df_long.drop(columns="Antibiotic_Metric", inplace=True)

## Restructuring Data for Visualization

The tidy dataset is reshaped into a structured wide format to facilitate visualization and statistical comparisons.

A pivot operation is performed using:

- Bacterial species, strain number, and antibiotic name as identifier variables
- Measurement types (Mean, SD, AST) as column categories
- Corresponding values stored in separate metric columns

This transformation produces a plotting-ready dataset where each row represents a unique isolate–antibiotic combination.

In [ ]:
# Pivot Table Creation
# Converting tidy long format to structured wide format
df_plot = (
    df_long
    .pivot_table(
        index=["Bacteria Name_Unnamed: 0_level_1", "Strain Number_Unnamed: 1_level_1", "Antibiotic"],
        columns="Metric",
        values="Value",
        aggfunc="first"  # If duplicates exist, take the first value. However, the dataset doesn't contain this type of value.
    )
    .reset_index()
)

df_plot.head()

## Data Type Standardization

Following the pivot operation, measurement columns may contain mixed data types due to the reshaping process.

To ensure consistency:

- Mean and standard deviation values are converted to numeric format
- Missing SD values are replaced with zero
- AST interpretations are retained as categorical text

These steps ensure that the dataset is suitable for statistical analysis and visualization.


In [ ]:
# Data type cleaning
df_plot["Mean"] = pd.to_numeric(df_plot["Mean"], errors="coerce")
df_plot["SD"] = pd.to_numeric(df_plot["SD"], errors="coerce").fillna(0) # Fill NaN with 0
df_plot["AST"] = df_plot["AST"].astype(str)

In [ ]:
# Print Column Names
print(df_plot.columns)

## Heatmap Visualization of Antimicrobial Susceptibility Variability

To visualize variability in antimicrobial susceptibility measurements, a heatmap is generated using the processed dataset.

The visualization displays:

- Rows representing individual bacterial isolates
- Columns representing tested antibiotics
- Color intensity corresponding to standard deviation values of inhibition zone diameters

A pivot operation is performed to structure the dataset into a matrix suitable for heatmap generation. The resulting visualization facilitates rapid identification of patterns in susceptibility variability across isolates and antibiotic classes.

Plot aesthetics, including color scales, axis formatting, and figure dimensions, are customized to enhance readability and ensure publication-quality presentation.


In [ ]:
# Import Plotly
import plotly.express as px

# Define the custom order for antibiotics
custom_antibiotic_order = [
    "Ampicillin (AMP)", "Cefotaxime (CTX)", "Ceftazidime (CAZ)", "Cefepime (FEP)",
    "Gentamycin (CN)", "Streptomycin (S)", "Tetracycline (TE)", "Chloramphenicol ©",
    "Ciprofloxacin (CIP)", "Levofloxacin (LEV)", "Nalidixic Acid (NA)",
    "Sulfametoxazole/Trimethoprim (SXT)", "Meropenem (MEM)", "Imipenem (IMI)", "Azithromycin (AZM)"
]

# Create Heatmap
fig_heatmap = px.imshow(
    df_plot.pivot_table(
        index="Strain Number_Unnamed: 1_level_1",
        columns="Antibiotic",
        values="SD" # Use Standard Devaition values
    ),
    color_continuous_scale="Viridis", # Definded  color pellete
    aspect="auto", # Adjust aspect ratio automatically
    title=None # No title
)

# Axis formatiing
fig_heatmap.update_xaxes(
    side="bottom", # Define x-axis side to bottom
    tickangle=90,
    title_text="Antibiotic",
    title_font=dict(color="black", family="Arial, sans-serif", weight="bold", size=14),
    tickfont=dict(color="black", family="Arial, sans-serif", weight="bold", size=12) # Add tickfont properties
)
fig_heatmap.update_yaxes(
    title_text="Strain Number",
    title_font=dict(color="black", family="Arial, sans-serif", weight="bold", size=14),
    tickfont=dict(color="black", family="Arial, sans-serif", weight="bold", size=12) # Add tickfont properties
)

# Layout Adjustment
fig_heatmap.update_layout(
    height=800,
    width=1200,
    margin=dict(l=100, r=100, t=100, b=100) # Adjust margins for better readability
)

# Display Plot
fig_heatmap.show()

## Multidimensional Visualization of Antimicrobial Susceptibility Patterns

A faceted scatter plot is generated to visualize antimicrobial susceptibility profiles across bacterial isolates.

The plot represents multiple dimensions simultaneously:

- **X-axis:** Antibiotic agents
- **Y-axis:** Mean inhibition zone diameters (mm)
- **Color:** AST interpretation category (Susceptible, Intermediate, Resistant)
- **Marker size:** Standard deviation of inhibition zone measurements
- **Facets:** Bacterial species

Antibiotics are displayed in a predefined clinical order to enhance interpretability.

This visualization enables comparative assessment of susceptibility patterns, variability, and resistance distribution across species and antibiotic classes.


In [ ]:
import plotly.express as px
import pandas as pd

# Define the custom order for antibiotics
custom_antibiotic_order = [
    "Ampicillin (AMP)", "Cefotaxime (CTX)", "Ceftazidime (CAZ)", "Cefepime (FEP)",
    "Gentamycin (CN)", "Streptomycin (S)", "Tetracycline (TE)", "Chloramphenicol ©",
    "Ciprofloxacin (CIP)", "Levofloxacin (LEV)", "Nalidixic Acid (NA)",
    "Sulfametoxazole/Trimethoprim (SXT)", "Meropenem (MEM)", "Imipenem (IMI)", "Azithromycin (AZM)"
]

# Convert the 'Antibiotic' column to a Categorical type with the custom order
df_plot['Antibiotic'] = pd.Categorical(
    df_plot['Antibiotic'],
    categories=custom_antibiotic_order,
    ordered=True
)

# Create scatter plot
fig = px.scatter(
    df_plot,
    x="Antibiotic",
    y="Mean",
    size="SD",
    color="AST",
    size_max=10, # Make bubbles smaller for better visualization
    opacity=0.75,
    color_discrete_map={
        "S": "#2CB1A1",
        "I": "#F5A623",
        "R": "#D64541"
    },
    hover_data={
        "Bacteria Name_Unnamed: 0_level_1": True,
        "Strain Number_Unnamed: 1_level_1": True,
        "Mean": ':.2f',
        "SD": ':.2f',
        "AST": True
    },
    category_orders={"Antibiotic": custom_antibiotic_order} # Explicitly set order for Plotly
)

fig.update_layout(
    template="plotly_white",
    xaxis_title={
        "text": "Antibiotic",
        "font": {
            "color": "black",
            "family": "Arial, sans-serif",
            "weight": "bold"
        }
    },
    yaxis_title={
        "text": "Mean <br> (mm)",
        "font": {
            "color": "black",
            "family": "Arial, sans-serif",
            "weight": "bold"
        }
    },
    legend_title={
        "text": "AST Interpretation",
        "font": {
            "color": "black",
            "family": "Arial, sans-serif",
            "weight": "bold"
        }
    },
    xaxis_tickangle=45,
    xaxis=dict(
        tickfont={
            "color": "black",
            "family": "Arial, sans-serif",
            "weight": "bold"
        }
    ),
    yaxis=dict(
        tickfont={
            "color": "black",
            "family": "Arial, sans-serif",
            "weight": "bold"
        },
        dtick=5 # Increase distance between y-axis tick labels
    ),
    legend=dict(
        font={
            "color": "black",
            "family": "Arial, sans-serif",
            "weight": "bold"
        }
    )
)

fig.update_traces(
    marker=dict(line=dict(width=0.5, color="white"))
)

fig.show()
